In [ ]:
# 若没装 fastkaggle 就安装并导入
# install fastkaggle if not available
try: import fastkaggle
except ModuleNotFoundError:
    !pip install -Uq fastkaggle

from fastkaggle import *

In [ ]:
# 设置竞赛、下载数据、导入 fastai、固定种子、取测试图片
comp = 'paddy-disease-classification'
path = setup_comp(comp, install='fastai "timm>=0.6.2.dev0"')
from fastai.vision.all import *
set_seed(42)

tst_files = get_image_files(path/'test_images').sorted()

In [ ]:
# 看各类别的样本数
df = pd.read_csv(path/'train.csv')
df.label.value_counts()

In [ ]:
# 先只用一个小类别的图片，让下面的显存测试跑得快
trn_path = path/'train_images'/'bacterial_panicle_blight'

In [ ]:
# 定义 train：支持梯度累积(accum)省显存；finetune=False 时只跑一轮测显存
def train(arch, size, item=Resize(480, method='squish'), accum=1, finetune=True, epochs=12):
    dls = ImageDataLoaders.from_folder(trn_path, valid_pct=0.2, item_tfms=item,
        batch_tfms=aug_transforms(size=size, min_scale=0.75), bs=64//accum)
    cbs = GradientAccumulation(64) if accum else []
    learn = vision_learner(dls, arch, metrics=error_rate, cbs=cbs).to_fp16()
    if finetune:
        learn.fine_tune(epochs, 0.01)
        return learn.tta(dl=dls.test_dl(tst_files))
    else:
        learn.unfreeze()
        learn.fit_one_cycle(epochs, 0.01)

In [ ]:
# 测 convnext_small（accum=1）的显存占用
train('convnext_small_in22k', 128, epochs=1, accum=1, finetune=False)

In [ ]:
# report_gpu：打印显存占用并清理缓存
import gc
def report_gpu():
    print(torch.cuda.list_gpu_processes())
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# 看显存
report_gpu()

In [ ]:
# accum=2 再测（显存应更低）
train('convnext_small_in22k', 128, epochs=1, accum=2, finetune=False)
report_gpu()

In [ ]:
# accum=4 再测
train('convnext_small_in22k', 128, epochs=1, accum=4, finetune=False)
report_gpu()

In [ ]:
# 测 convnext_large（更大模型）
train('convnext_large_in22k', 224, epochs=1, accum=2, finetune=False)
report_gpu()

In [ ]:
# convnext_large + 更大输入尺寸
train('convnext_large_in22k', (320,240), epochs=1, accum=2, finetune=False)
report_gpu()

In [ ]:
# 测 vit_large
train('vit_large_patch16_224', 224, epochs=1, accum=2, finetune=False)
report_gpu()

In [ ]:
# 测 swinv2_large
train('swinv2_large_window12_192_22k', 192, epochs=1, accum=2, finetune=False)
report_gpu()

In [ ]:
# 测 swin_large
train('swin_large_patch4_window7_224', 224, epochs=1, accum=2, finetune=False)
report_gpu()

In [ ]:
# 正式训练用的大分辨率 640×480
res = 640,480

In [ ]:
# 定义要集成的多个 “模型 × 输入尺寸” 组合
models = {
    'convnext_large_in22k': {
        (Resize(res), 224),
        (Resize(res), (320,224)),
    }, 'vit_large_patch16_224': {
        (Resize(480, method='squish'), 224),
        (Resize(res), 224),
    }, 'swinv2_large_window12_192_22k': {
        (Resize(480, method='squish'), 192),
        (Resize(res), 192),
    }, 'swin_large_patch4_window7_224': {
        (Resize(480, method='squish'), 224),
        (Resize(res), 224),
    }
}

In [ ]:
# 换回完整训练集路径
trn_path = path/'train_images'

In [ ]:
# 对每个组合训练并收集 TTA 预测
tta_res = []

for arch,details in models.items():
    for item,size in details:
        print('---',arch)
        print(size)
        print(item.name)
        tta_res.append(train(arch, size, item=item, accum=2)) #, epochs=1))
        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
# 把所有 TTA 结果存成 pickle
save_pickle('tta_res.pkl', tta_res)

In [ ]:
# 取出所有预测概率
tta_prs = first(zip(*tta_res))

In [ ]:
# 把 vit 的结果复制一份加权（它单独表现好）
tta_prs += tta_prs[2:4]

In [ ]:
# 所有预测取平均
avg_pr = torch.stack(tta_prs).mean(0)
avg_pr.shape

In [ ]:
# 建一个 DataLoaders 只为拿到类别词表 vocab
dls = ImageDataLoaders.from_folder(trn_path, valid_pct=0.2, item_tfms=Resize(480, method='squish'),
    batch_tfms=aug_transforms(size=224, min_scale=0.75))

In [ ]:
# 取最大概率类别，映射成名字，生成提交文件
idxs = avg_pr.argmax(dim=1)
vocab = np.array(dls.vocab)
ss = pd.read_csv(path/'sample_submission.csv')
ss['label'] = vocab[idxs]
ss.to_csv('subm.csv', index=False)

In [ ]:
# 提交到 Kaggle
if not iskaggle:
    from kaggle import api
    api.competition_submit_cli('subm.csv', 'part 3 v2', comp)

In [ ]:
# 推送 notebook 到 Kaggle（Jeremy 自用）
# This is what I use to push my notebook from my home PC to Kaggle

if not iskaggle:
    push_notebook('jhoward', 'scaling-up-road-to-the-top-part-3',
                  title='Scaling Up: Road to the Top, Part 3',
                  file='10-scaling-up-road-to-the-top-part-3.ipynb',
                  competition=comp, private=False, gpu=True)